# Multi-Omics Integration Pipeline

This notebook demonstrates part of my previous research work in multi-omics integration and computational biological data analysis using both Python and R workflows.

The analysis focuses on integrating RNA-seq and DNA methylation data for biologically meaningful feature analysis, biomarker discovery, and machine learning-based multi-omics integration.

The workflow includes:

* preprocessing and normalization,
* feature selection,
* biomarker analysis,
* transcriptomic and methylation integration,
* and heterogeneous biological data analysis for disease-related molecular interpretation.

This work reflects my broader research interest in:

* multi-omics integration,
* biologically informed AI,
* biomarker discovery,
* computational biology,
* and context-aware biological representation learning.


In [ ]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

Load Omic data for TCGA LUSC

In [ ]:
!pip install rpy2==3.5.1

In [ ]:
%load_ext rpy2.ipython

In [ ]:
#Set working directory
%%R
getwd()

Load Methylation data


In [ ]:
%%R
Meth = read.delim("/content/gdrive/MyDrive/data/Human__TCGA_LUSC__JHU_USC__Methylation__Meth450__01_28_2016__BI__Gene__Firehose_Methylation_Prepocessor.cct" ,header = TRUE, sep = "\t")

In [ ]:
%%R
knitr:: kable(head(Meth), "pipe", align = c("l", "c", "c") )

In [ ]:
%%R
%%Rdim(Meth)

In [ ]:
%%R
RNA= Meth

# Cleaning the data

## Data filtration

In [ ]:
%%R
## make features as a column
RNA_T= t(RNA)

In [ ]:
%%R
knitr:: kable(head(RNA_T), "pipe", align = c("l", "c", "c") )

In [ ]:
%%R
rowNms = RNA_T[ ,1]
colNms = RNA_T [1,]
RNA_T1 = RNA_T[-1,]


In [ ]:
%%R
colnames(RNA_T1) = colNms

In [ ]:
%%R

knitr:: kable(head(RNA_T1), "pipe", align = c("l", "c", "c") )

In [ ]:
%%R
head(rownames(RNA_T1))

remove features with zero variance`

In [ ]:
%%R
varCol_RNA=apply(RNA_T1, 2, var, na.rm = T)

In [ ]:
%%R
constCol_RNA <- (varCol_RNA == 0 | is.na(varCol_RNA))

In [ ]:
%%R
sum(constCol_RNA)

# Compute missinigness rate in your metaboloites

In [ ]:
%%R
h=hist((apply(is.na(RNA_T1), 2, sum)/nrow(RNA_T1) ) *100,breaks=10,
main="",
xlab="percentage of missingness")
text(h$mids,h$counts,labels=h$counts, adj=c(0.5, -0.5))
abline(v = 50)


In [ ]:
%%R
good.inx_RNA=apply(is.na(RNA_T1), 2, sum)/nrow(RNA_T1) <0.5
RNA1=RNA_T1[,good.inx_RNA]
dim(RNA1)

In [ ]:
%%R
write.csv(RNA_T1,"/content/gdrive/MyDrive/Meth_preprocessing1.csv")

# Impute missing values

# python

In [ ]:
import pandas as pd
df = pd.read_csv('/content/gdrive/MyDrive/Meth_preprocessing1.csv')

In [ ]:
df.head(13)

In [ ]:
import numpy as np
from sklearn.impute import KNNImputer

In [ ]:
nan = np.nan

In [ ]:
imputer = KNNImputer(n_neighbors=6, weights="uniform")

In [ ]:
df.iloc[: , 1:].head()

In [ ]:
df.iloc[: , 1:] = imputer.fit_transform(df.iloc[: , 1:])

In [ ]:
df.to_csv("/content/gdrive/MyDrive/Meth_preprocessing2.csv")

# Normalization
## Log transformation

# R

read files after imputing

In [ ]:
%%R
RNA = read.csv("/content/gdrive/MyDrive/Meth_preprocessing2.csv")

In [ ]:
%%R
knitr:: kable(head(RNA), "pipe", align = c("l", "c", "c") )

In [ ]:
%%R
RNA1= RNA[,-1:-2]

In [ ]:
%%R
RNA.logged <- log2(RNA1 + 1)

show distribution of methylation

In [ ]:
%%R
options(repr.plot.width=15,repr.plot.height=8)

par(mfrow=c(1,2))
plot(density( RNA1[,1]) ,main='befor log2')
plot(density( RNA.logged[,1]), main='after log2' )


par(mfrow=c(1,2))
plot(density( RNA1[,5]) ,main='befor log2')
plot(density( RNA.logged[,5]), main='after log2' )


In [ ]:
%%R
rownames(RNA.logged) = RNA[,2]
knitr:: kable(head(RNA.logged), "pipe", align = c("l", "c", "c") )


In [ ]:
%%R
write.csv(RNA.logged,"/content/gdrive/MyDrive/Meth_preprocessing3.csv")

Dist data using barplot

In [ ]:
%%R
options(repr.plot.width=10,repr.plot.height=8)
par(mar = c(8,5,2,2),mfrow=c(1,2))
boxplot(RNA1[,1:20], main="Before log2" ,horizontal=T, names=colnames(RNA1)[1:20],las=2,
       col = "lightgreen")
boxplot(RNA.logged[,1:20], main="After log2" ,horizontal=T, names=colnames(RNA.logged)[1:20],
        las=2,col = "lightgreen")


Scale

In [ ]:
%%R
knitr:: kable(head(RNA.logged), "pipe", align = c("l", "c", "c") )

In [ ]:
%%R
RNA.logged1 = RNA.logged

In [ ]:
%%R
RNA.logged.scaled=scale(RNA.logged1,center = TRUE, scale = TRUE)

In [ ]:
%%R
rownames(RNA.logged.scaled) = rownames(RNA.logged)

In [ ]:
%%R
knitr:: kable(head(RNA.logged.scaled), "pipe", align = c("l", "c", "c") )

In [ ]:
%%R
options(repr.plot.width=20,repr.plot.height=8)

par(mar = c(8,5,2,2),mfrow=c(1,3),cex.axis=1.5)
plot(density(apply(RNA1, 2, mean, na.rm = TRUE)),main='befor log2')
plot(density(apply(RNA.logged, 2, mean, na.rm = TRUE)),main='after log2')
plot(density(apply(RNA.logged.scaled, 2, mean, na.rm = TRUE)),main='after log2 + scaled')

In [ ]:
%%R
write.csv(RNA.logged.scaled,"/content/gdrive/Meth_preprocessing4.csv")

# Match stage

In [ ]:
import pandas as pd

In [ ]:
# Load the RNA file into a DataFrame

csv_df = pd.read_csv("/content/gdrive/MyDrive/Meth_preprocessing4.csv")

In [ ]:
# Load the clinical file into a DataFrame (assuming it's tab or space-separated)

txt_df = pd.read_csv("/content/gdrive/MyDrive/Clinical.txt", delimiter='\t')  # Adjust the delimiter as needed

In [ ]:
txt_df = txt_df.transpose()

In [ ]:
txt_df.head()

In [ ]:
# Use the first row as the header (column names)
txt_df.columns = txt_df.iloc[0]


In [ ]:
txt_df.head()

In [ ]:
txt_df.reset_index(inplace=True)

In [ ]:
txt_df.head()

In [ ]:
# Rename the first column to 'user ID'
txt_df.rename(columns={txt_df.columns[0]: 'user ID'}, inplace=True)

In [ ]:
txt_df.head()

In [ ]:
# Rename the first column in the CSV file to 'user ID'
csv_df.rename(columns={csv_df.columns[0]: 'user ID'}, inplace=True)

In [ ]:
csv_df.head()

In [ ]:
# Merge df1 with the 'stage' column from df2 based on 'user ID'
merged_df2 = pd.merge(csv_df, txt_df[['user ID', 'pathologic_stage']], on='user ID', how='left')

In [ ]:
merged_df2.head()

In [ ]:
merged_df2.to_csv("/content/gdrive/MyDrive/Meth_stage.csv")

In [ ]:
merged_df2.head()

In [ ]:
merged_df2['pathologic_stage'][0]

In [ ]:
merged_df2['pathologic_stage'][5]

In [ ]:
# Define a mapping function to standardize the stage names
def mapp_stage(stage):
    print("stage", stage)
    if pd.isna(stage):
        return None
    elif stage== 'stagei' :
        return 'Stage One'
    elif  stage== 'stageii' :
        return 'Stage Two'
    elif  stage== 'stageiii' :
        return 'Stage Three'
    elif  stage == 'stageiv' :
        return 'Stage Four'
    else:
        return stage


In [ ]:
merged_df3 = merged_df2

In [ ]:
# Apply the mapping function to the 'stage' column
merged_df3['pathologic_stage'] = merged_df3['pathologic_stage'].apply(mapp_stage)

In [ ]:
merged_df3['pathologic_stage']

In [ ]:
# Drop rows where the 'stage' column is null
merged_df3 = merged_df3.dropna(subset=['pathologic_stage'])

In [ ]:
merged_df3['pathologic_stage']

In [ ]:
# Count the occurrences of each stage
stage_counts = merged_df3['pathologic_stage'].value_counts()

In [ ]:
# Print the counts for each stage
print("Count of each stage:")
print(stage_counts)

split  70%  training
30 % testing

Stage One      244  : 170    74
Stage Two      162   : 113  49
Stage Three     84    : 58   26
stageiv          7  :    4  3

In [ ]:
merged_df3.head()

In [ ]:
merged_df3.to_csv("/content/gdrive/MyDrive/Meth_Stage2.csv")

In [ ]:
merged_df3.head()

code for stage 1 and two only

split train and test >> the meth_stage2 file for only stage 1 and 2

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the data (replace 'your_file.csv' with your actual file path)
# Example assumes a CSV file; adjust as necessary if using Excel or another format
data = pd.read_csv('/content/gdrive/MyDrive/Meth_Stage2.csv')  # If Excel, use pd.read_excel('your_file.xlsx')


In [ ]:
data = merged_df3

In [ ]:
# Filter for Stage One and Stage Two only
filtered_data = data[data['pathologic_stage'].isin(['Stage One', 'Stage Two'])]


In [ ]:
# Split the filtered data into training and testing sets (80% train, 20% test)
train_data, test_data = train_test_split(filtered_data, test_size=0.2, random_state=42)

# Save the split data to new files
train_data.to_csv('/content/gdrive/MyDrive/Meth_train_data_stage_one_two.csv', index=False)
test_data.to_csv('/content/gdrive/MyDrive/Meth_test_data_stage_one_two.csv', index=False)

In [ ]:
import pandas as pd
df1 = train_data
df2 = test_data

In [ ]:
df1.shape

In [ ]:
df2.shape

In [ ]:
# Filter out rows where the 'stage' column contains "Stage Three" or "Stage Four"
df1_filtered = df1[~df1['pathologic_stage'].isin(['Stage Three', 'Stage Four'])]
df2_filtered = df2[~df2['pathologic_stage'].isin(['Stage Three', 'Stage Four'])]

In [ ]:
df2_filtered.shape

In [ ]:
df1_filtered.shape

In [ ]:
df2_filtered.head()

In [ ]:
df1.head()

In [ ]:
df2.head()

#  XGBOOST

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split


In [ ]:
train_df = df1
test_df = df2

In [ ]:
# Separate features and target variable
X_train = train_df.drop(columns=['pathologic_stage'])
y_train = train_df['pathologic_stage']

In [ ]:
X_test = test_df.drop(columns=['pathologic_stage'])
y_test = test_df['pathologic_stage']

In [ ]:
# Encode the target variable
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [ ]:
# Convert the data into DMatrix format (optimized data structure for XGBoost)
dtrain = xgb.DMatrix(X_train.select_dtypes(include=['number', 'bool']), label=y_train_encoded, enable_categorical=True) # Select only numerical and boolean columns or set enable_categorical
dtest = xgb.DMatrix(X_test.select_dtypes(include=['number', 'bool']), label=y_test_encoded, enable_categorical=True) # Select only numerical and boolean columns or set enable_categorical

In [ ]:
# Define XGBoost parameters (these can be tuned)
params = {
    'objective': 'multi:softmax',  # Use 'multi:softprob' for probability outputs
    'num_class': len(label_encoder.classes_),  # Number of classes
    'eval_metric': 'mlogloss',  # Multiclass logloss
    'eta': 0.3,  # Learning rate
    'max_depth': 6,  # Maximum depth of a tree
    'subsample': 0.8,  # Subsample ratio of the training instances
    'colsample_bytree': 0.8  # Subsample ratio of columns when constructing each tree
}

In [ ]:
# Train the model
model = xgb.train(params, dtrain, num_boost_round=100)

In [ ]:
# Make predictions
y_pred = model.predict(dtest)

In [ ]:
# Convert predictions back to original labels
y_pred_labels = label_encoder.inverse_transform(y_pred.astype(int))

In [ ]:
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred_labels)
report = classification_report(y_test, y_pred_labels, target_names=label_encoder.classes_)



In [ ]:
print(f"Accuracy: {accuracy:.2f}")
print("Classification Report:")
print(report)

In [ ]:
import pandas as pd
import xgboost as xgb

# Assuming `model` is your trained model
# Get feature importance (e.g., gain, weight, or cover)
importance = model.get_score(importance_type='weight')  # You can use 'gain' or 'cover' as well

# Convert importance dictionary to DataFrame
importance_df = pd.DataFrame({
    'Feature': list(importance.keys()),
    'Importance': list(importance.values())
})

# Sort by importance (optional)
importance_df = importance_df.sort_values(by='Importance', ascending=False)




In [ ]:
len(importance_df)

In [ ]:
# Save the important features to a CSV file
importance_df.to_csv('/content/gdrive/MyDrive/XGboost_Imp628_Meth_stage1_2.csv', index=False)

In [ ]:
#save training and testing for stage 1 and 2 only
df1_filtered.to_csv('/content/gdrive/MyDrive/training_set_Meth_stage1_2.csv', index=False)
df2_filtered.to_csv('/content/gdrive/MyDrive/test_set_Meth_stage1_2.csv', index=False)

# **Integrate Methyaltion features and RNAseq features for deep learning**

In [ ]:
import pandas as pd

# Load the omics data files and important features lists
omics_data1 = pd.read_csv('/content/drive/MyDrive/RNA_Stage3.csv')  # RNA
omics_data2 = pd.read_csv('/content/drive/MyDrive/Meth_Stage2.csv')  # Methyaltion

In [ ]:
# Rename the second column to 'used_ID' in both datasets
omics_data1.columns.values[1] = 'used_ID'
omics_data2.columns.values[1] = 'used_ID'

In [ ]:
omics_data2.head()

In [ ]:
omics_data1.head()

In [ ]:
#check for stage one and two nly
# Step 2: Filter for "Stage One" and "Stage Two" only in each dataset
omics_data1_filtered = omics_data1[omics_data1['pathologic_stage'].isin(['Stage One', 'Stage Two'])]
omics_data2_filtered = omics_data2[omics_data2['pathologic_stage'].isin(['Stage One', 'Stage Two'])]

In [ ]:
omics_data2.shape

In [ ]:
omics_data2_filtered.shape

In [ ]:
omics_data1.shape

In [ ]:
omics_data1_filtered.shape

In [ ]:
# Step 3: Load important features for each omics data file
important_features1 = pd.read_csv('/content/drive/MyDrive/XGboost_Imp293_RNA_stage1_2.csv')['Feature'].tolist()  # Column 'Feature' with feature names
important_features2 = pd.read_csv('/content/drive/MyDrive/XGboost_Imp628_Meth_stage1_2.csv')['Feature'].tolist()  # Column 'Feature' with feature names


In [ ]:
# Step 4: Add 'user_id' and 'pathologic_stage' columns to the list of important features to retain them
important_features1 = ['used_ID', 'pathologic_stage'] + important_features1
important_features2 = ['used_ID', 'pathologic_stage'] + important_features2

In [ ]:
# Step 5: Filter each data file to include only the important features along with 'user_id' and 'pathologic_stage'
filtered_omics_data1 = omics_data1_filtered[important_features1]
filtered_omics_data2 = omics_data2_filtered[important_features2]

In [ ]:
filtered_omics_data1.shape

In [ ]:
filtered_omics_data2.shape

In [ ]:
filtered_omics_data1.head()

In [ ]:
filtered_omics_data2.head()

In [ ]:
# Step 6: Merge the two filtered data files on 'used_ID'
merged_data = pd.merge(filtered_omics_data1, filtered_omics_data2, on='used_ID', how='inner')


In [ ]:
merged_data.shape

In [ ]:
# Step 7: Save the final merged file
merged_data.to_csv('/content/drive/MyDrive/merged_final_RNA_Meth_data.csv', index=False)

print("Merged data saved to 'merged_filtered_omics_data.csv'")

#Deep Learning

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the merged omics data file
merged_data = pd.read_csv('/content/drive/MyDrive/merged_final_RNA_Meth_data.csv')  # Replace with your act

In [ ]:
# Drop the 'used_ID' column as it’s not relevant for model training
merged_data = merged_data.drop(columns=['used_ID' ,'pathologic_stage_y'])




In [ ]:
merged_data.head()

In [ ]:
# Define the target column and feature columns
# Assuming the 'pathologic_stage' column is the target for classification
target_column = 'pathologic_stage_x'
features = merged_data.drop(columns=[target_column])  # All columns except the target

In [ ]:
# Split data into features (X) and target (y)
X = features
y = merged_data[target_column]

In [ ]:
# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Optional: Save the split data to CSV files
X_train.to_csv('/content/drive/MyDrive/DL_X_train.csv', index=False)
X_test.to_csv('/content/drive/MyDrive/DL_X_test.csv', index=False)
y_train.to_csv('/content/drive/MyDrive/DL_y_train.csv', index=False)
y_test.to_csv('/content/drive/MyDrive/DL_y_test.csv', index=False)

print("Data split into train and test sets and saved to CSV files.")

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
training_array = X_train.values
test_array = X_test.values

In [ ]:
y_train = y_train.values
y_test = y_test.values

In [ ]:
print(training_array.shape)

In [ ]:
print(test_array.shape)

In [ ]:
# Reshape and normalize training data
trainX = training_array[:, 0:920].reshape(training_array.shape[0],1,23, 40).astype( 'float32' )


In [ ]:
# Reshape and normalize test data
testX = test_array[:, 0:920].reshape(test_array.shape[0],1,23,40).astype( 'float32' )

In [ ]:
#normalization
X_train = trainX

In [ ]:
#normalization
X_test = testX

In [ ]:
y_train

In [ ]:
y_test

In [ ]:
from sklearn import preprocessing
lb = preprocessing.LabelBinarizer()
y_train = lb.fit_transform(y_train)
y_test = lb.fit_transform(y_test)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Convolution2D, MaxPooling2D, Dropout, Flatten, Dense
import tensorflow.keras.backend as K

In [ ]:
model = Sequential()
K.set_image_data_format('channels_first')
model.add(Convolution2D(128,5, 5,  input_shape=(1,23, 40),activation= 'relu' , padding='same' ))
model.add(MaxPooling2D(pool_size=(2, 2)))
#model.add(Dropout(0.5))


#model.add(Convolution2D(64, 3, 3, activation= 'relu' , padding='same' ))
#model.add(MaxPooling2D(pool_size=(2, 2)))
#model.add(Dropout(0.5))

model.add(Convolution2D(32, 3, 3, activation= 'relu' , padding='same' ))
#model.add(MaxPooling2D(pool_size=(2, 2)))
#model.add(Dropout(0.5))

#model.add(Convolution2D(8, 3, 3, activation= 'relu' , padding='same' ))
#model.add(MaxPooling2D(pool_size=(2, 2)))
#model.add(Dropout(0.5))

model.add(Flatten())
model.add(Dense(1024, activation= 'relu' ))
model.add(Dense(500, activation= 'relu' ))
model.add(Dense(1, activation= 'sigmoid' ))
#model.add(Dropout(0.5))


In [ ]:
model.compile(loss= 'binary_crossentropy' , optimizer= 'adam' , metrics=[ 'accuracy' ])

In [ ]:
model.fit(X_train, y_train,
          epochs=100,
         )

In [ ]:
score = model.evaluate(X_test, y_test)

In [ ]:
model_json = model.to_json()
with open("/content/drive/MyDrive/LUSC_Model_RNA_Meth.json", "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
model.save_weights("/content/drive/MyDrive/LUSC_Model_RNA_Meth.weights.h5") # Added the .weights extension to the filename
print("Saved model to disk")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Predict the labels for the test data
y_pred = model.predict(X_test)


In [ ]:
# Predict the labels for the test data
y_pred = model.predict(X_test)
# Convert predicted probabilities to class labels (0 or 1)
y_pred_classes = (y_pred > 0.5).astype(int) # Assuming 0.5 as the threshold

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred_classes)) # Use y_pred_classes instead of y_pred

In [ ]:
# Classification Report
print("Classification Report:")
report = classification_report(y_test, y_pred_classes) # Store the classification report in the variable 'report'
print(report) # Use y_pred_classes instead of y_pred

# Save the report to a file
with open('/content/drive/MyDrive/LUSC_DL_RNA_Meth_classification_report.txt', 'w') as f:
    f.write(report)
print("Classification report saved to classification_report.txt")

In [ ]:
# Confusion Matrix
# Convert predicted probabilities to class labels (0 or 1)
y_pred_classes = (y_pred > 0.5).astype(int)  # Assuming 0.5 as the threshold

conf_matrix = confusion_matrix(y_test, y_pred_classes) #Use y_pred_classes for discrete labels

# Display the confusion matrix as a heatmap
plt.figure(figsize=(8, 6))
# Instead of model.classes_, use a list of your class labels
class_labels = [0, 1]  # Replace with your actual class labels if different
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
# Save the confusion matrix plot to a file
plt.savefig('/content/drive/MyDrive/LUSC_DL_RNA_Meth_confusion_matrix.png')

plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt

# ... (rest of your code) ...

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred)  # Use y_pred (probabilities) for ROC curve
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
# Save the ROC curve plot
plt.savefig('/content/drive/MyDrive/LUSC_DL_RNA_Meth_roc_curve.png')

plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
import seaborn as sns
import matplotlib.pyplot as plt

# ... (rest of your code) ...

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred)
average_precision = average_precision_score(y_test, y_pred)

plt.figure()
plt.step(recall, precision, color='b', alpha=0.2, where='post')
plt.fill_between(recall, precision, step='post', alpha=0.2, color='b')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.title('Precision-Recall curve: AP={0:0.2f}'.format(average_precision))

# Save the Precision-Recall curve plot
plt.savefig('/content/drive/MyDrive/LUSC_DL_RNA_Meth_precision_recall_curve.png')

plt.show()

need to know the integrated file (RNA and MEthyaltuin ... how many stage 1 and how mant stage 2

In [ ]:
import pandas as pd

# Read the merged data file
merged_data = pd.read_csv('/content/drive/MyDrive/merged_final_RNA_Meth_data.csv')

# Assuming there is a column named 'stage' in the data
# Replace 'stage' with the actual column name containing stage information
stage_counts = merged_data['pathologic_stage_y'].value_counts()

print("Number of stage 1 and stage 2 entries:")
print(stage_counts[['Stage One', 'Stage Two']])

# Conclusion

This work reflects my research interest in multi-omics integration, biologically informed feature selection, biomarker discovery, and AI-driven computational biology.

The presented workflow demonstrates the integration of heterogeneous biological datasets using both Python and R for biologically meaningful molecular analysis and disease-related interpretation.

The experience gained from this work motivates my current interest in developing next-generation context-aware and multi-scale foundation AI models integrating transcriptomics, single-cell RNA sequencing, spatial transcriptomics, and histopathology imaging for biologically interpretable disease modeling and cross-modal biomarker discovery.
